In [ ]:
import numpy as np
from scipy.optimize import minimize

def rqr_pointwise_loss(y, mu1, mu2, alpha):
    kappa = (y - mu1) * (y - mu2)
    loss = np.where(
        kappa >= 0,
        alpha * kappa,
        (alpha - 1) * kappa
    )
    return loss

def rqr_empirical_loss(params, y, alpha):
    mu1, mu2 = params
    loss = rqr_pointwise_loss(y, mu1, mu2, alpha)
    return np.mean(loss)

def fit_rqr(y, alpha=0.5, init=None):
    if init is None:
        init = [np.quantile(y, (1 - alpha) / 2), np.quantile(y, 1 - (1 - alpha) / 2)]
    result = minimize(
        fun=rqr_empirical_loss,
        x0=init,
        args=(y, alpha),
        method='BFGS'
    )
    mu1_star, mu2_star = np.sort(result.x)  # Ensure mu1 < mu2
    return mu1_star, mu2_star, result

# 1. Simulate data
np.random.seed(0)
n = 10000
y = np.random.normal(0, 1, size=n)

# 2. Fit RQR
alpha = 0.95
mu1_star, mu2_star, result = fit_rqr(y, alpha=alpha)

# 3. Compute empirical in-sample coverage
coverage = np.mean((y > mu1_star) & (y < mu2_star))

print(f"alpha: {alpha}")
print(f"mu1*: {mu1_star:.4f}, mu2*: {mu2_star:.4f}")
print(f"Empirical coverage: {coverage:.4f}")


In [ ]:
import numpy as np
from itertools import combinations
from tqdm import tqdm

def compute_rqr_loss(y, a, b, alpha):
    """
    Compute the sum of the RQR loss for interval (a, b) over data y.
    """
    loss = 0.0
    for yi in y:
        kappa = (yi - a) * (yi - b)
        if kappa >= 0:
            loss += alpha * kappa
        else:
            loss += (alpha - 1) * kappa
    return loss

def compute_log_posterior(y, a, b, alpha, w):
    """
    Compute the unnormalized log-posterior for a given interval (a, b).
    """
    return -w * compute_rqr_loss(y, a, b, alpha)

def find_map_rqr_interval(y, alpha, w=1.0, grid_size=200):
    """
    Grid search to find (a, b) with a < b that maximizes the log-posterior.
    """
    y_sorted = np.sort(y)
    a_vals = np.linspace(y_sorted[0] - 1, y_sorted[-1], grid_size)
    b_vals = np.linspace(y_sorted[0], y_sorted[-1] + 1, grid_size)
    max_log_post = -np.inf
    best_a, best_b = None, None
    
    # Use meshgrid and vectorized search for speed
    A, B = np.meshgrid(a_vals, b_vals)
    A_flat, B_flat = A.flatten(), B.flatten()
    mask = A_flat < B_flat
    A_flat, B_flat = A_flat[mask], B_flat[mask]

    for a, b in tqdm(zip(A_flat, B_flat), total=len(A_flat), desc="Searching intervals"):
        log_post = compute_log_posterior(y, a, b, alpha, w)
        if log_post > max_log_post:
            max_log_post = log_post
            best_a, best_b = a, b

    return best_a, best_b, max_log_post

def empirical_coverage(y, a, b):
    """
    Compute the empirical in-sample coverage for interval (a, b).
    """
    count = np.sum((y > min(a, b)) & (y < max(a, b)))
    return count / len(y)

# ----- Example usage -----
if __name__ == "__main__":
    np.random.seed(42)
    # Generate sample data
    n = 200
    y = np.random.normal(loc=0, scale=1, size=n)
    alpha = 0.7
    w = 1.0
    
    # Find MAP/MPE interval
    a_hat, b_hat, log_post = find_map_rqr_interval(y, alpha, w, grid_size=150)
    print(f"MAP/MPE Interval: ({a_hat:.4f}, {b_hat:.4f})")
    
    # Empirical coverage
    coverage = empirical_coverage(y, a_hat, b_hat)
    print(f"Empirical Coverage: {coverage:.4f} (Target alpha: {alpha})")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit  # sigmoid
from tqdm import trange

# --------- RQR Loss and Posterior Log-Kernel ---------

def rqr_loss(y, a, b, alpha):
    """RQR loss for all y, scalar a, b."""
    kappa = (y - a) * (y - b)
    loss = np.where(kappa >= 0, alpha * kappa, (alpha - 1) * kappa)
    return np.sum(loss)

def log_posterior(y, a, b, alpha, w):
    """Log-posterior (unnormalized) for (a, b)."""
    if a >= b:
        return -np.inf
    return -w * rqr_loss(y, a, b, alpha)

def grad_log_posterior(y, a, b, alpha, w):
    """Gradient of log-posterior with respect to a and b."""
    # Avoid evaluating at a >= b
    if a >= b:
        return np.array([0.0, 0.0])
    kappa = (y - a) * (y - b)
    # d/d a: - (y-b)
    # d/d b: - (y-a)
    grad_a = np.where(kappa >= 0,
                      alpha * (-(y - b)),
                      (alpha - 1) * (-(y - b)))
    grad_b = np.where(kappa >= 0,
                      alpha * (-(y - a)),
                      (alpha - 1) * (-(y - a)))
    return -w * np.array([np.sum(grad_a), np.sum(grad_b)])

# --------- HMC Sampler with Constraint a < b ---------

def unconstrained_to_ab(x, y):
    """Transform unconstrained (x, y) to (a, b) with a < b."""
    a = x
    b = x + np.exp(y)  # always b > a
    return a, b

def ab_to_unconstrained(a, b):
    """Inverse transform: (a, b) -> (x, y) for a < b."""
    x = a
    y = np.log(b - a)
    return x, y

def log_posterior_unconstrained(u, y, alpha, w):
    a, b = unconstrained_to_ab(u[0], u[1])
    # Add log-Jacobian from transformation (b - a)
    return log_posterior(y, a, b, alpha, w) + u[1]

def grad_log_posterior_unconstrained(u, y, alpha, w, eps=1e-8):
    # Chain rule
    a, b = unconstrained_to_ab(u[0], u[1])
    grad = grad_log_posterior(y, a, b, alpha, w)
    # d a / d x = 1, d a / d y = 0
    # d b / d x = 1, d b / d y = exp(y)
    d_a_dx = 1
    d_a_dy = 0
    d_b_dx = 1
    d_b_dy = np.exp(u[1])
    # Chain rule
    dL_dx = grad[0] * d_a_dx + grad[1] * d_b_dx
    dL_dy = grad[0] * d_a_dy + grad[1] * d_b_dy
    # Add log-Jacobian term derivative
    dL_dy += 1
    return np.array([dL_dx, dL_dy])

def hmc_sampler(logp, grad_logp, init, step_size, n_steps, n_samples, y, alpha, w, burnin=500, thin=5):
    samples = []
    u = np.array(init)
    for i in trange(n_samples * thin + burnin):
        # HMC leapfrog
        p = np.random.normal(size=2)
        current_u = np.copy(u)
        current_p = np.copy(p)
        # Half-step momentum
        p += 0.5 * step_size * grad_logp(u, y, alpha, w)
        for _ in range(n_steps):
            u += step_size * p
            if _ != n_steps - 1:
                p += step_size * grad_logp(u, y, alpha, w)
        # Final half-step
        p += 0.5 * step_size * grad_logp(u, y, alpha, w)
        p = -p  # Negate for symmetry
        # MH accept
        current_logp = logp(current_u, y, alpha, w)
        proposed_logp = logp(u, y, alpha, w)
        accept_prob = np.exp(proposed_logp - current_logp + 0.5 * (np.sum(current_p ** 2) - np.sum(p ** 2)))
        if np.random.rand() < accept_prob:
            accepted = True
        else:
            u = current_u  # reject
            accepted = False
        if i >= burnin and (i - burnin) % thin == 0:
            samples.append(u.copy())
    return np.array(samples)

# --------- Posterior summary and coverage ---------

def posterior_summary(samples):
    """Transform to (a, b), compute means."""
    a_vals, b_vals = unconstrained_to_ab(samples[:,0], samples[:,1])
    mean_a = np.mean(a_vals)
    mean_b = np.mean(b_vals)
    return mean_a, mean_b, a_vals, b_vals

def posterior_mean_coverage(y, a_vals, b_vals):
    """Empirical mean posterior coverage (average over samples)."""
    coverages = []
    for a, b in zip(a_vals, b_vals):
        coverages.append(np.mean((y > a) & (y < b)))
    return np.mean(coverages)

# --------- Main Demo ---------

if __name__ == "__main__":
    np.random.seed(42)
    n = 2000
    y = np.random.normal(0, 1, n)
    alpha = 0.95
    w = 1.0

    # Start near sample quantiles
    q_lo, q_hi = np.quantile(y, [(1-alpha)/2, 1-(1-alpha)/2])
    init_a = q_lo
    init_b = q_hi
    init = ab_to_unconstrained(init_a, init_b)
    
    print("Starting HMC sampling...")
    samples = hmc_sampler(
        logp=log_posterior_unconstrained,
        grad_logp=grad_log_posterior_unconstrained,
        init=init,
        step_size=0.003,
        n_steps=10,
        n_samples=2000,
        y=y,
        alpha=alpha,
        w=w,
        burnin=600,
        thin=5
    )
    


In [ ]:
    mean_a, mean_b, a_vals, b_vals = posterior_summary(samples)
    print(f"Posterior mean interval: ({mean_a:.4f}, {mean_b:.4f})")

    # Empirical posterior mean coverage
    post_cov = posterior_mean_coverage(y, a_vals, b_vals)
    print(f"Posterior mean empirical coverage: {post_cov:.4f} (Target alpha: {alpha})")
    
    # Trace and diagnostics
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(a_vals, label='a (lower)')
    plt.plot(b_vals, label='b (upper)')
    plt.xlabel("Iteration")
    plt.ylabel("Interval Bound")
    plt.title("HMC Traces")
    plt.legend()

    plt.subplot(1,2,2)
    plt.acorr(a_vals - np.mean(a_vals), maxlags=50, normed=True, lw=2, color='tab:blue', label='a autocorr')
    plt.acorr(b_vals - np.mean(b_vals), maxlags=50, normed=True, lw=2, color='tab:orange', label='b autocorr')
    plt.title("Autocorrelation of Chains")
    plt.xlabel("Lag")
    plt.tight_layout()

    import seaborn as sns


    # --- Compute 95% credible intervals for a and b ---
    ci_a_lower, ci_a_upper = np.percentile(a_vals, [2.5, 97.5])
    ci_b_lower, ci_b_upper = np.percentile(b_vals, [2.5, 97.5])

    # --- Histogram with Posterior Mean Interval and Credible Bounds ---
    plt.figure(figsize=(8,5))
    sns.histplot(y, bins=30, color='skyblue', kde=True, stat="density", edgecolor='k')
    plt.axvline(mean_a, color='red', linestyle='--', linewidth=2, label='Posterior Mean Lower')
    plt.axvline(mean_b, color='red', linestyle='--', linewidth=2, label='Posterior Mean Upper')

    # 95% credible intervals for a and b
    plt.axvline(ci_a_lower, color='purple', linestyle=':', linewidth=2, label='95% CrI Lower')
    plt.axvline(ci_a_upper, color='purple', linestyle=':', linewidth=2)
    plt.axvline(ci_b_lower, color='orange', linestyle=':', linewidth=2, label='95% CrI Upper')
    plt.axvline(ci_b_upper, color='orange', linestyle=':', linewidth=2)

    plt.title(f"Histogram of $y$ with Posterior Mean {alpha:.2f} Interval\nand 95% Credible Intervals for Bounds")
    plt.xlabel("y")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()


    plt.show()
